In [1]:
import numpy as np 
import polars as pl 
import polars.selectors as cs
import pandas as pd 
import utils

In [2]:
# import data
train = pl.read_parquet("../data/cleaned/application_train.parquet")
test  = pl.read_parquet("../data/cleaned/application_test.parquet")
buro  = pl.read_parquet("../data/cleaned/bureau.parquet")
bbal  = pl.read_parquet("../data/cleaned/bureau_balance.parquet")
prev  = pl.read_parquet("../data/cleaned/previous_application.parquet")
card  = pl.read_parquet("../data/cleaned/credit_card_balance.parquet")
poca  = pl.read_parquet("../data/cleaned/POS_CASH_balance.parquet")
inst  = pl.read_parquet("../data/cleaned/installments_payments.parquet")

In [3]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [4]:
# garbage collection
import gc
gc.enable()

In [5]:
# check dimensions
print("Application:", train.shape, test.shape)
print("Buro:", buro.shape)
print("Bbal:", bbal.shape)
print("Prev:", prev.shape)
print("Card:", card.shape)
print("Poca:", poca.shape)
print("Inst:", inst.shape)

Application: (307511, 122) (48744, 121)
Buro: (1716428, 17)
Bbal: (27299925, 3)
Prev: (1670214, 37)
Card: (3840312, 23)
Poca: (10001358, 8)
Inst: (13605401, 8)


In [6]:
# extract target
y = train.select(["SK_ID_CURR", "TARGET"])
train = train.drop("TARGET")

In [7]:
# concatenate application data
appl = pl.concat([train, test])
del train, test

In [8]:
import polars as pl
import utils

# list of documents
doc_vars = ["FLAG_DOCUMENT_2",  "FLAG_DOCUMENT_3",  "FLAG_DOCUMENT_4",  "FLAG_DOCUMENT_5",  "FLAG_DOCUMENT_6",
            "FLAG_DOCUMENT_7",  "FLAG_DOCUMENT_8",  "FLAG_DOCUMENT_9",  "FLAG_DOCUMENT_10", "FLAG_DOCUMENT_11",
            "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13", "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15", "FLAG_DOCUMENT_16",
            "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_18", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"]

# 1. Feature Engineering (computed in parallel)
appl = appl.with_columns(
    # income ratios
    (pl.col("AMT_CREDIT") / pl.col("AMT_INCOME_TOTAL")).alias("CREDIT_BY_INCOME"),
    (pl.col("AMT_ANNUITY") / pl.col("AMT_INCOME_TOTAL")).alias("ANNUITY_BY_INCOME"),
    (pl.col("AMT_GOODS_PRICE") / pl.col("AMT_INCOME_TOTAL")).alias("GOODS_PRICE_BY_INCOME"),
    (pl.col("AMT_INCOME_TOTAL") / pl.col("CNT_FAM_MEMBERS")).alias("INCOME_PER_PERSON"),
    
    # career ratio (replaces negatives with None)
    pl.when((pl.col("DAYS_EMPLOYED") / pl.col("DAYS_BIRTH")) < 0)
      .then(None)
      .otherwise(pl.col("DAYS_EMPLOYED") / pl.col("DAYS_BIRTH"))
      .alias("PERCENT_WORKED"),
      
    # number of adults and children ratio
    (pl.col("CNT_FAM_MEMBERS") - pl.col("CNT_CHILDREN")).alias("CNT_ADULTS"),
    (pl.col("CNT_CHILDREN") / pl.col("CNT_FAM_MEMBERS")).alias("CHILDREN_RATIO"),
    
    # overall payments
    (pl.col("AMT_CREDIT") / pl.col("AMT_ANNUITY")).alias("ANNUITY LENGTH"),
    
    # external sources
    pl.mean_horizontal("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3").alias("EXT_SOURCE_MEAN"),
    # sum_horizontal on is_not_null() perfectly mimics the "3 - sum(is_null)" logic natively
    pl.sum_horizontal(
        pl.col("EXT_SOURCE_1").is_not_null(),
        pl.col("EXT_SOURCE_2").is_not_null(),
        pl.col("EXT_SOURCE_3").is_not_null()
    ).alias("NUM_EXT_SOURCES"),
    
    # number of documents
    pl.sum_horizontal(doc_vars).alias("NUM_DOCUMENTS"),
    
    # application date (Weekend / Working day)
    pl.when(pl.col("WEEKDAY_APPR_PROCESS_START").is_in(["SATURDAY", "SUNDAY"]))
      .then(pl.lit("Weekend"))
      .otherwise(pl.lit("Working day"))
      .alias("DAY_APPR_PROCESS_START"),
      
    # age ratios
    (pl.col("OWN_CAR_AGE") / pl.col("DAYS_BIRTH")).alias("OWN_CAR_AGE_RATIO"),
    (pl.col("DAYS_ID_PUBLISH") / pl.col("DAYS_BIRTH")).alias("DAYS_ID_PUBLISHED_RATIO"),
    (pl.col("DAYS_REGISTRATION") / pl.col("DAYS_BIRTH")).alias("DAYS_REGISTRATION_RATIO"),
    (pl.col("DAYS_LAST_PHONE_CHANGE") / pl.col("DAYS_BIRTH")).alias("DAYS_LAST_PHONE_CHANGE_RATIO")
)

# 2. Apply Custom Functions (These now work correctly with the Polars DataFrame)
log_vars = ["AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_GOODS_PRICE", "AMT_ANNUITY"]
appl = utils.create_logarithms(appl, log_vars, replace=True)

day_vars = ["DAYS_BIRTH", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_EMPLOYED", "DAYS_LAST_PHONE_CHANGE"]
appl = utils.convert_days(appl, day_vars, t=30, rounding=True, replace=True)

# 3. Drop unused features
drops = ['APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 
         'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
         'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
         'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'COMMONAREA_MODE','ELEVATORS_MODE', 'ENTRANCES_MODE', 
         'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 
         'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'TOTALAREA_MODE',  'YEARS_BEGINEXPLUATATION_MODE']

appl = appl.drop(drops)


In [9]:
rename_mapping = {col: f"app_{col}" for col in appl.columns if col != "SK_ID_CURR"}
appl.rename(rename_mapping)

SK_ID_CURR,app_NAME_CONTRACT_TYPE,app_CODE_GENDER,app_FLAG_OWN_CAR,app_FLAG_OWN_REALTY,app_CNT_CHILDREN,app_AMT_INCOME_TOTAL,app_AMT_CREDIT,app_AMT_ANNUITY,app_AMT_GOODS_PRICE,app_NAME_TYPE_SUITE,app_NAME_INCOME_TYPE,app_NAME_EDUCATION_TYPE,app_NAME_FAMILY_STATUS,app_NAME_HOUSING_TYPE,app_REGION_POPULATION_RELATIVE,app_DAYS_BIRTH,app_DAYS_EMPLOYED,app_DAYS_REGISTRATION,app_DAYS_ID_PUBLISH,app_OWN_CAR_AGE,app_FLAG_MOBIL,app_FLAG_EMP_PHONE,app_FLAG_WORK_PHONE,app_FLAG_CONT_MOBILE,app_FLAG_PHONE,app_FLAG_EMAIL,app_OCCUPATION_TYPE,app_CNT_FAM_MEMBERS,app_REGION_RATING_CLIENT,app_REGION_RATING_CLIENT_W_CITY,app_WEEKDAY_APPR_PROCESS_START,app_HOUR_APPR_PROCESS_START,app_REG_REGION_NOT_LIVE_REGION,app_REG_REGION_NOT_WORK_REGION,app_LIVE_REGION_NOT_WORK_REGION,app_REG_CITY_NOT_LIVE_CITY,…,app_FLAG_DOCUMENT_7,app_FLAG_DOCUMENT_8,app_FLAG_DOCUMENT_9,app_FLAG_DOCUMENT_10,app_FLAG_DOCUMENT_11,app_FLAG_DOCUMENT_12,app_FLAG_DOCUMENT_13,app_FLAG_DOCUMENT_14,app_FLAG_DOCUMENT_15,app_FLAG_DOCUMENT_16,app_FLAG_DOCUMENT_17,app_FLAG_DOCUMENT_18,app_FLAG_DOCUMENT_19,app_FLAG_DOCUMENT_20,app_FLAG_DOCUMENT_21,app_AMT_REQ_CREDIT_BUREAU_HOUR,app_AMT_REQ_CREDIT_BUREAU_DAY,app_AMT_REQ_CREDIT_BUREAU_WEEK,app_AMT_REQ_CREDIT_BUREAU_MON,app_AMT_REQ_CREDIT_BUREAU_QRT,app_AMT_REQ_CREDIT_BUREAU_YEAR,app_CREDIT_BY_INCOME,app_ANNUITY_BY_INCOME,app_GOODS_PRICE_BY_INCOME,app_INCOME_PER_PERSON,app_PERCENT_WORKED,app_CNT_ADULTS,app_CHILDREN_RATIO,app_ANNUITY LENGTH,app_EXT_SOURCE_MEAN,app_NUM_EXT_SOURCES,app_NUM_DOCUMENTS,app_DAY_APPR_PROCESS_START,app_OWN_CAR_AGE_RATIO,app_DAYS_ID_PUBLISHED_RATIO,app_DAYS_REGISTRATION_RATIO,app_DAYS_LAST_PHONE_CHANGE_RATIO
i32,str,str,str,str,i32,f32,f32,f32,f32,str,str,str,str,str,f32,f64,f64,f32,f64,f32,i32,i32,i32,i32,i32,i32,str,f32,i32,i32,str,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,u32,i32,str,f64,f64,f64,f64
100002,"""Cash loans""","""M""","""N""","""Y""",0,12.218501,12.915583,10.11462,12.768545,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.018801,315.0,21.0,122.0,71.0,null,1,1,0,1,1,0,"""Laborers""",1.0,2,2,"""WEDNESDAY""",10,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2.007889,0.121978,1.733333,202500.0,0.067329,1.0,0.0,16.461103,0.161787,3,1,"""Working day""",null,0.224078,0.385583,0.11986
100003,"""Cash loans""","""F""","""N""","""N""",0,12.506182,14.072865,10.482893,13.937287,"""Family""","""State servant""","""Higher education""","""Married""","""House / apartment""",0.003541,559.0,40.0,40.0,10.0,null,1,1,0,1,1,0,"""Core staff""",2.0,1,1,"""MONDAY""",11,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.79075,0.132217,4.183333,135000.0,0.070862,2.0,0.0,36.234085,0.466757,2,1,"""Working day""",null,0.017358,0.070743,0.049389
100004,"""Revolving loans""","""M""","""Y""","""Y""",0,11.119899,11.813039,8.817447,11.813039,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.010032,635.0,8.0,142.0,84.0,26.0,1,1,1,1,1,0,"""Laborers""",1.0,2,2,"""MONDAY""",9,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.1,2.0,67500.0,0.011814,1.0,0.0,20.0,0.642739,2,0,"""Working day""",-0.001365,0.132889,0.223669,0.042791
100006,"""Cash loans""","""F""","""N""","""Y""",0,11.813039,12.652947,10.298482,12.601492,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Civil marriage""","""House / apartment""",0.008019,634.0,101.0,328.0,81.0,null,1,1,0,1,0,0,"""Laborers""",2.0,2,2,"""WEDNESDAY""",17,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,2.316167,0.2199,2.2,67500.0,0.159905,2.0,0.0,10.532818,0.650442,1,1,"""Working day""",null,0.128229,0.51739,0.032465
100007,"""Cash loans""","""M""","""N""","""Y""",0,11.707679,13.148034,9.992712,13.148034,"""Unaccompanied""","""Working""","""Secondary / secondary s

In [10]:
# check data
appl.head()

SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,…,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,CREDIT_BY_INCOME,ANNUITY_BY_INCOME,GOODS_PRICE_BY_INCOME,INCOME_PER_PERSON,PERCENT_WORKED,CNT_ADULTS,CHILDREN_RATIO,ANNUITY LENGTH,EXT_SOURCE_MEAN,NUM_EXT_SOURCES,NUM_DOCUMENTS,DAY_APPR_PROCESS_START,OWN_CAR_AGE_RATIO,DAYS_ID_PUBLISHED_RATIO,DAYS_REGISTRATION_RATIO,DAYS_LAST_PHONE_CHANGE_RATIO
i32,str,str,str,str,i32,f32,f32,f32,f32,str,str,str,str,str,f32,f64,f64,f32,f64,f32,i32,i32,i32,i32,i32,i32,str,f32,i32,i32,str,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,u32,i32,str,f64,f64,f64,f64
100002,"""Cash loans""","""M""","""N""","""Y""",0,12.218501,12.915583,10.11462,12.768545,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.018801,315.0,21.0,122.0,71.0,null,1,1,0,1,1,0,"""Laborers""",1.0,2,2,"""WEDNESDAY""",10,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2.007889,0.121978,1.733333,202500.0,0.067329,1.0,0.0,16.461103,0.161787,3,1,"""Working day""",null,0.224078,0.385583,0.11986
100003,"""Cash loans""","""F""","""N""","""N""",0,12.506182,14.072865,10.482893,13.937287,"""Family""","""State servant""","""Higher education""","""Married""","""House / apartment""",0.003541,559.0,40.0,40.0,10.0,null,1,1,0,1,1,0,"""Core staff""",2.0,1,1,"""MONDAY""",11,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.79075,0.132217,4.183333,135000.0,0.070862,2.0,0.0,36.234085,0.466757,2,1,"""Working day""",null,0.017358,0.070743,0.049389
100004,"""Revolving loans""","""M""","""Y""","""Y""",0,11.119899,11.813039,8.817447,11.813039,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.010032,635.0,8.0,142.0,84.0,26.0,1,1,1,1,1,0,"""Laborers""",1.0,2,2,"""MONDAY""",9,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.1,2.0,67500.0,0.011814,1.0,0.0,20.0,0.642739,2,0,"""Working day""",-0.001365,0.132889,0.223669,0.042791
100006,"""Cash loans""","""F""","""N""","""Y""",0,11.813039,12.652947,10.298482,12.601492,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Civil marriage""","""House / apartment""",0.008019,634.0,101.0,328.0,81.0,null,1,1,0,1,0,0,"""Laborers""",2.0,2,2,"""WEDNESDAY""",17,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,2.316167,0.2199,2.2,67500.0,0.159905,2.0,0.0,10.532818,0.650442,1,1,"""Working day""",null,0.128229,0.51739,0.032465
100007,"""Cash loans""","""M""","""N""","""Y""",0,11.707679,13.148034,9.992712,13.148034,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.028663,664.0,101.0,144.0,115.0,null,1,1,0,1,0,0,"""Core staff""",1.0,2,2,"""THURSDAY""",11,0,0,0,0,…,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.222222,0.179963,4.222222,121500.0,0.152418,1.0,0.0,23.461618,0.322738,1,

In [11]:

train_base = appl.filter(pl.col("SK_ID_CURR").is_in(y["SK_ID_CURR"]))
test_base = appl.filter(~pl.col("SK_ID_CURR").is_in(y["SK_ID_CURR"]))


train_base.write_parquet("../data/processed/train_base.parquet")
test_base.write_parquet("../data/processed/test_base.parquet")
y.write_parquet("../data/processed/y_base.parquet")
print(f"Base Train Shape: {train_base.shape}")
print(f"Base Test Shape: {test_base.shape}")


Base Train Shape: (307511, 109)
Base Test Shape: (48744, 109)


In [11]:
# count missings
nas = utils.count_missings(appl)
nas.head()

Feature,Total,Percent
str,i64,f64
"""COMMONAREA_AVG""",248360,69.714109
"""NONLIVINGAPARTMENTS_AVG""",246861,69.293343
"""FONDKAPREMONT_MODE""",243092,68.235393
"""LIVINGAPARTMENTS_AVG""",242979,68.203674
"""FLOORSMIN_AVG""",241108,67.678489


In [12]:
# check bbal data
bbal.head()

SK_ID_BUREAU,MONTHS_BALANCE,STATUS
i32,i32,str
5715448,0,"""C"""
5715448,-1,"""C"""
5715448,-2,"""C"""
5715448,-3,"""C"""
5715448,-4,"""C"""


In [13]:
loan_score = (
    bbal
    .with_columns(
            pl.col("STATUS").replace({"X": None, "1": 1.0, "2": 2.0, "3": 3.0, "4": 4.0, "5": 5.0}, default=0.0
    ).alias("NUM_STATUS")
    )
    .with_columns(
        (pl.col("NUM_STATUS")/(pl.col("MONTHS_BALANCE").abs()+1)).alias("LOAN_SCORE")
    )
    .group_by("SK_ID_BUREAU")
    .agg(pl.col("LOAN_SCORE").sum())
)

bbal = bbal.to_dummies("STATUS")

In [14]:
# count missings
nas = utils.count_missings(bbal)
nas.head()

Feature,Total,Percent
str,i64,f64


In [15]:
agg_bbal = (
    bbal
    .group_by("SK_ID_BUREAU")
    .agg(
        pl.col("MONTHS_BALANCE").count().alias("MONTH_COUNT"),
        cs.starts_with("STATUS_").mean()
    )
    .join(loan_score,on= "SK_ID_BUREAU",how = "left")
)

In [16]:
# count missings
nas = utils.count_missings(agg_bbal)
nas.head()

Feature,Total,Percent
str,i64,f64


In [17]:
# check data
agg_bbal.head()

SK_ID_BUREAU,MONTH_COUNT,STATUS_0,STATUS_1,STATUS_2,STATUS_3,STATUS_4,STATUS_5,STATUS_C,STATUS_X,LOAN_SCORE
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64
5863477,26,0.115385,0.0,0.0,0.0,0.0,0.0,0.0,0.884615,0.0
5439053,46,0.0,0.0,0.0,0.0,0.0,0.0,0.217391,0.782609,0.0
5539203,12,0.333333,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,0.0
6290149,89,0.011236,0.0,0.0,0.0,0.0,0.0,0.898876,0.089888,0.0
6388840,6,0.833333,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0


In [18]:
# clear memory
del bbal

In [19]:
# check buro data
buro.head()

SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
i32,i32,str,str,i32,i32,f32,f32,f32,i32,f32,f32,f32,f32,str,i32,f32
215354,5714462,"""Closed""","""currency 1""",-497,0,-153.0,-153.0,null,0,91323.0,0.0,null,0.0,"""Consumer credit""",-131,null
215354,5714463,"""Active""","""currency 1""",-208,0,1075.0,null,null,0,225000.0,171342.0,null,0.0,"""Credit card""",-20,null
215354,5714464,"""Active""","""currency 1""",-203,0,528.0,null,null,0,464323.5,null,null,0.0,"""Consumer credit""",-16,null
215354,5714465,"""Active""","""currency 1""",-203,0,null,null,null,0,90000.0,null,null,0.0,"""Credit card""",-16,null
215354,5714466,"""Active""","""currency 1""",-629,0,1197.0,null,77674.5,0,2.7e6,null,null,0.0,"""Consumer credit""",-21,null


In [20]:
buro = buro.join(agg_bbal, how = "left", on = "SK_ID_BUREAU")

In [21]:
# Total bureau loans per applicant
buro = buro.with_columns(
    pl.len().over("SK_ID_CURR").alias("CNT_BURO_LOANS")
)


buro = buro.with_columns(
    (pl.col("AMT_CREDIT_SUM_OVERDUE") / pl.col("AMT_ANNUITY")).alias("AMT_SUM_OVERDUE_RATIO_1"),
    (pl.col("AMT_CREDIT_SUM_OVERDUE") / pl.col("AMT_CREDIT_SUM")).alias("AMT_SUM_OVERDUE_RATIO_2"),
    (pl.col("AMT_CREDIT_MAX_OVERDUE") / pl.col("AMT_ANNUITY")).alias("AMT_MAX_OVERDUE_RATIO_1"),
    (pl.col("AMT_CREDIT_MAX_OVERDUE") / pl.col("AMT_CREDIT_SUM")).alias("AMT_MAX_OVERDUE_RATIO_2"),
    (pl.col("AMT_CREDIT_SUM_DEBT") / pl.col("AMT_CREDIT_SUM")).alias("AMT_SUM_DEBT_RATIO_1"),
    (pl.col("AMT_CREDIT_SUM_DEBT") / pl.col("AMT_CREDIT_SUM_LIMIT")).alias("AMT_SUM_DEBT_RATIO_2"),
)

log_vars = ["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
buro = utils.create_logarithms(buro, log_vars, replace=True)

# Convert Days
day_vars = ["DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE"]
buro = utils.convert_days(buro, day_vars, t=1, rounding=False, replace=True)

# 5. Remaining engineered features
buro = buro.with_columns(
    # Recency-weighted loan score
    (pl.col("LOAN_SCORE") / (pl.col("DAYS_CREDIT") / 12)).alias("WEIGHTED_LOAN_SCORE"),
    
    (pl.col("DAYS_ENDDATE_FACT") - pl.col("DAYS_CREDIT_ENDDATE")).alias("DAYS_END_DIFF_1"),
    (pl.col("DAYS_CREDIT_UPDATE") - pl.col("DAYS_CREDIT_ENDDATE")).alias("DAYS_END_DIFF_2"),
    (pl.col("DAYS_CREDIT_ENDDATE") - pl.col("DAYS_CREDIT")).alias("DAYS_DURATION_1"),
    (pl.col("DAYS_ENDDATE_FACT") - pl.col("DAYS_CREDIT")).alias("DAYS_DURATION_2"),
    
    (pl.col("CREDIT_ACTIVE") == "Active").sum().over("SK_ID_CURR").alias("CNT_BURO_ACTIVE"),
    (pl.col("CREDIT_ACTIVE") == "Closed").sum().over("SK_ID_CURR").alias("CNT_BURO_CLOSED"),
    (pl.col("CREDIT_ACTIVE") == "Bad debt").sum().over("SK_ID_CURR").alias("CNT_BURO_BAD")
)


In [22]:
buro = buro.to_dummies(cs.string(),drop_first=True)

In [23]:
# count missings
nas = utils.count_missings(buro)
nas.head()

Feature,Total,Percent
str,i64,f64
"""AMT_MAX_OVERDUE_RATIO_1""",1598872,93.151125
"""AMT_ANNUITY""",1226791,71.47349
"""AMT_SUM_OVERDUE_RATIO_1""",1226791,71.47349
"""AMT_CREDIT_MAX_OVERDUE""",1124488,65.513264
"""AMT_MAX_OVERDUE_RATIO_2""",1124488,65.513264


In [24]:
cnt_buro = (
    buro
    .group_by("SK_ID_CURR")
    .agg(
        pl.col("SK_ID_BUREAU").count().alias("buro_BURO_COUNT")
    )
)
buro = buro.drop("SK_ID_BUREAU")

agg_buro = utils.aggregate_data(buro,id_var= "SK_ID_CURR",label = "buro")

agg_buro = agg_buro.join(cnt_buro,on="SK_ID_CURR",how="left")

agg_buro = agg_buro.drop([
    "buro_WEIGHTED_LOAN_SCORE_std", 
    "buro_WEIGHTED_LOAN_SCORE_min", 
    "buro_WEIGHTED_LOAN_SCORE_max"
])

- Preparing the dataset...
- Extracted 0 factors and 57 numerics...
- Aggregating numeric features...
- Final dimensions: (305811, 229)


In [25]:
# count missings
nas = utils.count_missings(agg_buro)
nas.head()

Feature,Total,Percent
str,i64,f64
"""buro_AMT_MAX_OVERDUE_RATIO_1_s…",277289,90.673324
"""buro_AMT_MAX_OVERDUE_RATIO_1_m…",242976,79.452995
"""buro_AMT_MAX_OVERDUE_RATIO_1_m…",242976,79.452995
"""buro_AMT_MAX_OVERDUE_RATIO_1_m…",242976,79.452995
"""buro_AMT_ANNUITY_std""",213412,69.785587


In [29]:
# check data
print(agg_buro.shape)
agg_buro.head()


(305811, 227)


SK_ID_CURR,buro_CREDIT_ACTIVE_Active_mean,buro_CREDIT_ACTIVE_Bad debt_mean,buro_CREDIT_ACTIVE_Sold_mean,buro_CREDIT_CURRENCY_currency 2_mean,buro_CREDIT_CURRENCY_currency 3_mean,buro_CREDIT_CURRENCY_currency 4_mean,buro_DAYS_CREDIT_mean,buro_CREDIT_DAY_OVERDUE_mean,buro_DAYS_CREDIT_ENDDATE_mean,buro_DAYS_ENDDATE_FACT_mean,buro_AMT_CREDIT_MAX_OVERDUE_mean,buro_CNT_CREDIT_PROLONG_mean,buro_AMT_CREDIT_SUM_mean,buro_AMT_CREDIT_SUM_DEBT_mean,buro_AMT_CREDIT_SUM_LIMIT_mean,buro_AMT_CREDIT_SUM_OVERDUE_mean,buro_CREDIT_TYPE_Another type of loan_mean,buro_CREDIT_TYPE_Car loan_mean,buro_CREDIT_TYPE_Cash loan (non-earmarked)_mean,buro_CREDIT_TYPE_Credit card_mean,buro_CREDIT_TYPE_Interbank credit_mean,buro_CREDIT_TYPE_Loan for business development_mean,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_mean,buro_CREDIT_TYPE_Loan for the purchase of equipment_mean,buro_CREDIT_TYPE_Loan for working capital replenishment_mean,buro_CREDIT_TYPE_Microloan_mean,buro_CREDIT_TYPE_Mobile operator loan_mean,buro_CREDIT_TYPE_Mortgage_mean,buro_CREDIT_TYPE_Real estate loan_mean,buro_CREDIT_TYPE_Unknown type of loan_mean,buro_DAYS_CREDIT_UPDATE_mean,buro_AMT_ANNUITY_mean,buro_MONTH_COUNT_mean,buro_STATUS_0_mean,buro_STATUS_1_mean,buro_STATUS_2_mean,…,buro_CREDIT_TYPE_Interbank credit_max,buro_CREDIT_TYPE_Loan for business development_max,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_max,buro_CREDIT_TYPE_Loan for the purchase of equipment_max,buro_CREDIT_TYPE_Loan for working capital replenishment_max,buro_CREDIT_TYPE_Microloan_max,buro_CREDIT_TYPE_Mobile operator loan_max,buro_CREDIT_TYPE_Mortgage_max,buro_CREDIT_TYPE_Real estate loan_max,buro_CREDIT_TYPE_Unknown type of loan_max,buro_DAYS_CREDIT_UPDATE_max,buro_AMT_ANNUITY_max,buro_MONTH_COUNT_max,buro_STATUS_0_max,buro_STATUS_1_max,buro_STATUS_2_max,buro_STATUS_3_max,buro_STATUS_4_max,buro_STATUS_5_max,buro_STATUS_C_max,buro_STATUS_X_max,buro_LOAN_SCORE_max,buro_CNT_BURO_LOANS_max,buro_AMT_SUM_OVERDUE_RATIO_1_max,buro_AMT_SUM_OVERDUE_RATIO_2_max,buro_AMT_MAX_OVERDUE_RATIO_1_max,buro_AMT_MAX_OVERDUE_RATIO_2_max,buro_AMT_SUM_DEBT_RATIO_1_max,buro_AMT_SUM_DEBT_RATIO_2_max,buro_DAYS_END_DIFF_1_max,buro_DAYS_END_DIFF_2_max,buro_DAYS_DURATION_1_max,buro_DAYS_DURATION_2_max,buro_CNT_BURO_ACTIVE_max,buro_CNT_BURO_CLOSED_max,buro_CNT_BURO_BAD_max,buro_BURO_COUNT
i32,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f64,f64,f64,f64,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f64,f32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,u32,u32,u32,u32
370592,0.4,0.0,0.0,0.0,0.0,0.0,1413.2,0.0,1772.0,1770.0,0.0,0.0,9.983135,2.632029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,943.0,8.677248,46.8,0.410312,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,2272.0,8.912744,84,1.0,0.0,0.0,0.0,0.0,0.0,0.97619,0.411765,0.0,5,0.0,0.0,0.0,0.0,0.87933,NaN,93.0,93.0,-91.0,-63.0,2,3,0,5
391695,0.0,0.0,0.0,0.0,0.0,0.0,2270.0,0.0,444.0,430.0,null,0.0,12.100718,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,391.0,null,null,null,null,null,…,0,0,0,0,0,0,0,0,0,0,391.0,null,null,null,null,null,null,null,null,null,null,null,1,null,0.0,null,null,0.0,NaN,-14.0,-53.0,-1826.0,-1840.0,0,1,0,1
138435,0.571429,0.0,0.0,0.0,0.0,0.0,820.142857,0.0,405.5,458.333344,null,0.0,11.959614,4.500415,0.0,0.0,0.0,0.0,0.0,0.142857,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,249.714286,2.127156,21.428571,0.48567,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,856.0,7.445044,35,0.916667,0.0,0.0,0.0,0.0,0.0,0.857143,0.529412,0.0,7,0.0,0.0,null,null,0.872419,inf,212.0,211.0,-365.0,-153.0,4,3,0,7
179331,0.214286,0.0,0.0,0.0,0.0,0.0,1117.0,0.0,1647.0,636.272705,1897.275024,0.0,10.856487,1.253323,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,350.0,0.0,26.142857,0.469388,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,2150.0,0.0,70,1.0,0.0,0.0,0.0,0.0,0.0,0.842857,1.0,0.0,14,NaN,0.0,inf,0.019984,1.0,NaN,36.0,0.0,-304.0,-216.0,3,11,0,14
44

In [30]:
del buro

In [31]:
# check inst data
inst.head()

SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
i32,i32,f32,i32,f32,f32,f32,f32
1054186,161674,1.0,6,-1180.0,-1187.0,6948.359863,6948.359863
1330831,151639,0.0,34,-2156.0,-2156.0,1716.525024,1716.525024
2085231,193053,2.0,1,-63.0,-63.0,25425.0,25425.0
2452527,199697,1.0,3,-2418.0,-2426.0,24350.130859,24350.130859
2714724,167756,1.0,2,-1383.0,-1366.0,2165.040039,2160.584961


In [32]:
inst = (
    inst
    .with_columns(
        (pl.col("DAYS_ENTRY_PAYMENT")-pl.col("DAYS_INSTALMENT")).clip(lower_bound=0).alias("DPD"),
        (pl.col("DAYS_INSTALMENT")-pl.col("DAYS_ENTRY_PAYMENT")).clip(lower_bound=0).alias("DBD"),
        (pl.col("AMT_PAYMENT")/pl.col("AMT_INSTALMENT")).alias("PAYMENT_PERC"),
        (pl.col("AMT_INSTALMENT")-pl.col("AMT_PAYMENT")).alias("PAYMENT_DIFF")
    )
)

# logarithms
log_vars = ["AMT_INSTALMENT", "AMT_PAYMENT"]
inst = utils.create_logarithms(inst, log_vars, replace = True)

In [33]:
inst = inst.to_dummies(cs.string(),drop_first=True)

In [34]:
# count missings
nas = utils.count_missings(inst)
nas.head()

Feature,Total,Percent
str,i64,f64
"""DAYS_ENTRY_PAYMENT""",2905,0.021352
"""AMT_PAYMENT""",2905,0.021352
"""DPD""",2905,0.021352
"""DBD""",2905,0.021352
"""PAYMENT_PERC""",2905,0.021352


In [35]:
### AGGREGATIONS

inst_id = inst.select(["SK_ID_CURR", "SK_ID_PREV"]).unique()


cnt_inst = (
    inst
    .group_by("SK_ID_PREV")
    .agg(
        pl.col("NUM_INSTALMENT_NUMBER").count().alias("inst_INST_COUNT")
    )
)

inst = inst.drop(["NUM_INSTALMENT_NUMBER","SK_ID_CURR"])

agg_inst = utils.aggregate_data(inst,id_var = "SK_ID_PREV")

agg_inst = agg_inst.join(cnt_inst,how="left",on = "SK_ID_PREV")

agg_inst = agg_inst.join(inst_id,on = "SK_ID_PREV", how= "left")
agg_inst = agg_inst.drop("SK_ID_PREV")

agg_inst = utils.aggregate_data(agg_inst,id_var = "SK_ID_CURR",label="inst")



- Preparing the dataset...
- Extracted 0 factors and 9 numerics...
- Aggregating numeric features...
- Final dimensions: (997752, 37)
- Preparing the dataset...
- Extracted 0 factors and 37 numerics...
- Aggregating numeric features...
- Final dimensions: (339587, 149)


In [36]:
# count missings
nas = utils.count_missings(agg_inst)
nas.head()

Feature,Total,Percent
str,i64,f64
"""inst_DAYS_ENTRY_PAYMENT_std_st…",99700,29.359192
"""inst_AMT_PAYMENT_std_std""",99700,29.359192
"""inst_DPD_std_std""",99700,29.359192
"""inst_DBD_std_std""",99700,29.359192
"""inst_PAYMENT_PERC_std_std""",99700,29.359192


In [37]:
# check data
agg_inst.head()

SK_ID_CURR,inst_NUM_INSTALMENT_VERSION_mean_mean,inst_DAYS_INSTALMENT_mean_mean,inst_DAYS_ENTRY_PAYMENT_mean_mean,inst_AMT_INSTALMENT_mean_mean,inst_AMT_PAYMENT_mean_mean,inst_DPD_mean_mean,inst_DBD_mean_mean,inst_PAYMENT_PERC_mean_mean,inst_PAYMENT_DIFF_mean_mean,inst_NUM_INSTALMENT_VERSION_std_mean,inst_DAYS_INSTALMENT_std_mean,inst_DAYS_ENTRY_PAYMENT_std_mean,inst_AMT_INSTALMENT_std_mean,inst_AMT_PAYMENT_std_mean,inst_DPD_std_mean,inst_DBD_std_mean,inst_PAYMENT_PERC_std_mean,inst_PAYMENT_DIFF_std_mean,inst_NUM_INSTALMENT_VERSION_min_mean,inst_DAYS_INSTALMENT_min_mean,inst_DAYS_ENTRY_PAYMENT_min_mean,inst_AMT_INSTALMENT_min_mean,inst_AMT_PAYMENT_min_mean,inst_DPD_min_mean,inst_DBD_min_mean,inst_PAYMENT_PERC_min_mean,inst_PAYMENT_DIFF_min_mean,inst_NUM_INSTALMENT_VERSION_max_mean,inst_DAYS_INSTALMENT_max_mean,inst_DAYS_ENTRY_PAYMENT_max_mean,inst_AMT_INSTALMENT_max_mean,inst_AMT_PAYMENT_max_mean,inst_DPD_max_mean,inst_DBD_max_mean,inst_PAYMENT_PERC_max_mean,inst_PAYMENT_DIFF_max_mean,…,inst_NUM_INSTALMENT_VERSION_mean_max,inst_DAYS_INSTALMENT_mean_max,inst_DAYS_ENTRY_PAYMENT_mean_max,inst_AMT_INSTALMENT_mean_max,inst_AMT_PAYMENT_mean_max,inst_DPD_mean_max,inst_DBD_mean_max,inst_PAYMENT_PERC_mean_max,inst_PAYMENT_DIFF_mean_max,inst_NUM_INSTALMENT_VERSION_std_max,inst_DAYS_INSTALMENT_std_max,inst_DAYS_ENTRY_PAYMENT_std_max,inst_AMT_INSTALMENT_std_max,inst_AMT_PAYMENT_std_max,inst_DPD_std_max,inst_DBD_std_max,inst_PAYMENT_PERC_std_max,inst_PAYMENT_DIFF_std_max,inst_NUM_INSTALMENT_VERSION_min_max,inst_DAYS_INSTALMENT_min_max,inst_DAYS_ENTRY_PAYMENT_min_max,inst_AMT_INSTALMENT_min_max,inst_AMT_PAYMENT_min_max,inst_DPD_min_max,inst_DBD_min_max,inst_PAYMENT_PERC_min_max,inst_PAYMENT_DIFF_min_max,inst_NUM_INSTALMENT_VERSION_max_max,inst_DAYS_INSTALMENT_max_max,inst_DAYS_ENTRY_PAYMENT_max_max,inst_AMT_INSTALMENT_max_max,inst_AMT_PAYMENT_max_max,inst_DPD_max_max,inst_DBD_max_max,inst_PAYMENT_PERC_max_max,inst_PAYMENT_DIFF_max_max,inst_inst_INST_COUNT_max
i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,u32
141463,1.090909,-282.818176,-293.59198,9.484498,9.546226,0.0,10.773809,1.171866,-324.703644,0.215557,88.258041,88.007561,0.170409,0.033432,0.0,4.883143,0.550177,909.489075,1.0,-416.0,-424.333344,8.974693,9.471264,0.0,3.333333,1.0,-3021.344971,1.666667,-156.0,-172.333328,9.542562,9.59834,0.0,19.0,2.829804,0.0,…,1.272727,-120.0,-132.571426,9.866901,9.866901,0.0,12.571428,1.515598,0.0,0.64667,108.166542,106.565491,0.510858,0.099926,0.0,6.451283,1.650532,2728.467285,1.0,-210.0,-222.0,9.866901,9.866901,0.0,6.0,1.0,0.0,3.0,-30.0,-47.0,9.866901,9.866901,0.0,20.0,6.489412,0.0,12
240678,0.73183,-503.905792,-532.220215,8.559084,8.535544,0.0,28.314426,0.992754,33.976627,0.20246,143.31723,146.827682,0.801904,0.823449,0.0,20.225233,0.047166,221.151367,0.666667,-732.666687,-742.0,7.313683,7.313683,0.0,3.333333,0.680087,0.0,1.333333,-274.333344,-283.666656,10.698656,10.698656,0.0,63.666668,1.0,1500.0,…,1.142857,-139.0,-157.571426,8.824508,8.824508,0.0,62.263157,1.0,101.929886,0.377964,196.32486,197.479828,1.332839,1.397475,0.0,38.964146,0.141498,663.454102,1.0,-229.0,-238.0,8.632367,8.632367,0.0,6.0,1.0,0.0,2.0,-26.0,-29.0,11.356662,11.356662,0.0,131.0,1.0,4500.0,46
406093,1.0,-1472.0,-1483.0,8.387589,8.387589,0.0,11.0,1.0,0.0,0.0,56.124863,58.415752,0.000364,0.000364,0.0,3.224903,0.0,0.0,1.0,-1547.0,-1559.0,8.386847,8.386847,0.0,5.0,1.0,0.0,1.0,-1397.0,-1402.0,8.387738,8.387738,0.0,14.0,1.0,0.0,…,1.0,-1472.0,-1483.0,8.387589,8.387589,0.0,11.0,1.0,0.0,0.0,56.124863,58.415752,0.000364,0.000364,0.0,3.224903,0.0,0.0,1.0,-1547.0,-1559.0,8.386847,8.386847,0.0,5.0,1.0,0.0,1.0,-1397.0,-1402.0,8.387738,8.387738,0.0,14.0,1.0,0.0,6
149850,1.0,-2677.0,-2680.416748,8.734055,8.734055,0.333333,3.75,1.0,0.0,0.0,108.166542,

In [38]:
# clear memory
del inst

In [39]:
# check poca data
poca.head()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i32,i32,i32,f32,f32,str,i32,i32
1803195,182943,-31,48.0,45.0,"""Active""",0,0
1715348,367990,-33,36.0,35.0,"""Active""",0,0
1784872,397406,-32,12.0,9.0,"""Active""",0,0
1903291,269225,-35,48.0,42.0,"""Active""",0,0
2341044,334279,-35,36.0,35.0,"""Active""",0,0


In [40]:
### FEATURE ENGINEERING

# installments percentage

poca = (
    poca
    .with_columns(
        (pl.col("CNT_INSTALMENT_FUTURE")/pl.col("CNT_INSTALMENT")).alias("INSTALLMENTS_PERCENT")
    )
)


In [42]:
poca = poca.to_dummies(cs.string(),drop_first=True)

In [43]:
# count missings
nas = utils.count_missings(poca)
nas.head()

Feature,Total,Percent
str,i64,f64
"""INSTALLMENTS_PERCENT""",26184,0.261804
"""CNT_INSTALMENT_FUTURE""",26087,0.260835
"""CNT_INSTALMENT""",26071,0.260675


In [44]:
### AGGREGATIONS
poca_id  = poca.select(["SK_ID_CURR", "SK_ID_PREV"]).unique()
cnt_mon = (
    poca
    .group_by("SK_ID_PREV")
    .agg(
        pl.col("MONTHS_BALANCE").count().alias("poca_MON_COUNT")
    )
)

poca = poca.drop(["MONTHS_BALANCE","SK_ID_CURR"])


# aggregate data
agg_poca = utils.aggregate_data(poca, id_var = "SK_ID_PREV")

agg_poca = agg_poca.join(cnt_mon,how="left",on = "SK_ID_PREV")
agg_poca = agg_poca.join(poca_id,how="left",on = "SK_ID_PREV")

agg_poca = agg_poca.drop("SK_ID_PREV")

agg_poca = utils.aggregate_data(agg_poca, id_var = "SK_ID_CURR", label = "poca")


- Preparing the dataset...
- Extracted 0 factors and 13 numerics...
- Aggregating numeric features...
- Final dimensions: (936325, 53)
- Preparing the dataset...
- Extracted 0 factors and 53 numerics...
- Aggregating numeric features...
- Final dimensions: (337252, 213)


In [45]:
# count missings
nas = utils.count_missings(agg_poca)
nas.head()

Feature,Total,Percent
str,i64,f64
"""poca_CNT_INSTALMENT_std_std""",106269,31.510265
"""poca_CNT_INSTALMENT_FUTURE_std…",106269,31.510265
"""poca_INSTALLMENTS_PERCENT_std_…",106269,31.510265
"""poca_NAME_CONTRACT_STATUS_Amor…",106037,31.441474
"""poca_NAME_CONTRACT_STATUS_Appr…",106037,31.441474


In [46]:
# check data
agg_poca.head()

SK_ID_CURR,poca_CNT_INSTALMENT_mean_mean,poca_CNT_INSTALMENT_FUTURE_mean_mean,poca_NAME_CONTRACT_STATUS_Amortized debt_mean_mean,poca_NAME_CONTRACT_STATUS_Approved_mean_mean,poca_NAME_CONTRACT_STATUS_Canceled_mean_mean,poca_NAME_CONTRACT_STATUS_Completed_mean_mean,poca_NAME_CONTRACT_STATUS_Demand_mean_mean,poca_NAME_CONTRACT_STATUS_Returned to the store_mean_mean,poca_NAME_CONTRACT_STATUS_Signed_mean_mean,poca_NAME_CONTRACT_STATUS_XNA_mean_mean,poca_SK_DPD_mean_mean,poca_SK_DPD_DEF_mean_mean,poca_INSTALLMENTS_PERCENT_mean_mean,poca_CNT_INSTALMENT_std_mean,poca_CNT_INSTALMENT_FUTURE_std_mean,poca_NAME_CONTRACT_STATUS_Amortized debt_std_mean,poca_NAME_CONTRACT_STATUS_Approved_std_mean,poca_NAME_CONTRACT_STATUS_Canceled_std_mean,poca_NAME_CONTRACT_STATUS_Completed_std_mean,poca_NAME_CONTRACT_STATUS_Demand_std_mean,poca_NAME_CONTRACT_STATUS_Returned to the store_std_mean,poca_NAME_CONTRACT_STATUS_Signed_std_mean,poca_NAME_CONTRACT_STATUS_XNA_std_mean,poca_SK_DPD_std_mean,poca_SK_DPD_DEF_std_mean,poca_INSTALLMENTS_PERCENT_std_mean,poca_CNT_INSTALMENT_min_mean,poca_CNT_INSTALMENT_FUTURE_min_mean,poca_NAME_CONTRACT_STATUS_Amortized debt_min_mean,poca_NAME_CONTRACT_STATUS_Approved_min_mean,poca_NAME_CONTRACT_STATUS_Canceled_min_mean,poca_NAME_CONTRACT_STATUS_Completed_min_mean,poca_NAME_CONTRACT_STATUS_Demand_min_mean,poca_NAME_CONTRACT_STATUS_Returned to the store_min_mean,poca_NAME_CONTRACT_STATUS_Signed_min_mean,poca_NAME_CONTRACT_STATUS_XNA_min_mean,…,poca_NAME_CONTRACT_STATUS_Approved_std_max,poca_NAME_CONTRACT_STATUS_Canceled_std_max,poca_NAME_CONTRACT_STATUS_Completed_std_max,poca_NAME_CONTRACT_STATUS_Demand_std_max,poca_NAME_CONTRACT_STATUS_Returned to the store_std_max,poca_NAME_CONTRACT_STATUS_Signed_std_max,poca_NAME_CONTRACT_STATUS_XNA_std_max,poca_SK_DPD_std_max,poca_SK_DPD_DEF_std_max,poca_INSTALLMENTS_PERCENT_std_max,poca_CNT_INSTALMENT_min_max,poca_CNT_INSTALMENT_FUTURE_min_max,poca_NAME_CONTRACT_STATUS_Amortized debt_min_max,poca_NAME_CONTRACT_STATUS_Approved_min_max,poca_NAME_CONTRACT_STATUS_Canceled_min_max,poca_NAME_CONTRACT_STATUS_Completed_min_max,poca_NAME_CONTRACT_STATUS_Demand_min_max,poca_NAME_CONTRACT_STATUS_Returned to the store_min_max,poca_NAME_CONTRACT_STATUS_Signed_min_max,poca_NAME_CONTRACT_STATUS_XNA_min_max,poca_SK_DPD_min_max,poca_SK_DPD_DEF_min_max,poca_INSTALLMENTS_PERCENT_min_max,poca_CNT_INSTALMENT_max_max,poca_CNT_INSTALMENT_FUTURE_max_max,poca_NAME_CONTRACT_STATUS_Amortized debt_max_max,poca_NAME_CONTRACT_STATUS_Approved_max_max,poca_NAME_CONTRACT_STATUS_Canceled_max_max,poca_NAME_CONTRACT_STATUS_Completed_max_max,poca_NAME_CONTRACT_STATUS_Demand_max_max,poca_NAME_CONTRACT_STATUS_Returned to the store_max_max,poca_NAME_CONTRACT_STATUS_Signed_max_max,poca_NAME_CONTRACT_STATUS_XNA_max_max,poca_SK_DPD_max_max,poca_SK_DPD_DEF_max_max,poca_INSTALLMENTS_PERCENT_max_max,poca_poca_MON_COUNT_max
i32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,u8,u8,u8,u8,u8,u8,u8,u8,i32,i32,f32,f32,f32,u8,u8,u8,u8,u8,u8,u8,u8,i32,i32,f32,u32
134517,10.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.7,0.0,2.160247,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.216025,10.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.216025,10.0,4.0,0,0,0,0,0,0,0,0,0,0,0.4,10.0,10.0,0,0,0,0,0,0,0,0,0,0,1.0,7
103568,14.371428,10.103572,0.0,0.0,0.0,0.08428,0.0,0.0,0.0,0.0,0.0,0.0,0.581746,2.044405,4.86099,0.0,0.0,0.0,0.241257,0.0,0.0,0.0,0.0,0.0,0.0,0.318408,10.714286,2.285714,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.447214,0.0,0.0,0.0,0.0,0.0,0.0,0.429703,24.0,15.0,0,0,0,0,0,0,0,0,0,0,0.625,36.0,36.0,0,0,0,1,0,0,0,0,0,0,1.0,13
136854,5.6,3.6,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.894427,2.302173,0.0,0.0,0.0,0.447214,0.0,0.0,0.0,0.0,0.0,0.0,0.383695,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.447214,0.0,0.0,0.0,0.0,0.0,0.0,0.383695,4.0,0.0,0,0,0,0,0

In [47]:
# clear memory
del poca

In [48]:
# check card data
card.head()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i32,i32,i32,f32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,str,i32,i32
2562384,378907,-6,56.970001,135000,0.0,877.5,0.0,877.5,1700.324951,1800.0,1800.0,0.0,0.0,0.0,0.0,1,0.0,1.0,35.0,"""Active""",0,0
2582071,363914,-1,63975.554688,45000,2250.0,2250.0,0.0,0.0,2250.0,2250.0,2250.0,60175.078125,64875.554688,64875.554688,1.0,1,0.0,0.0,69.0,"""Active""",0,0
1740877,371185,-7,31815.224609,450000,0.0,0.0,0.0,0.0,2250.0,2250.0,2250.0,26926.425781,31460.085938,31460.085938,0.0,0,0.0,0.0,30.0,"""Active""",0,0
1389973,337855,-4,236572.109375,225000,2250.0,2250.0,0.0,0.0,11795.759766,11925.0,11925.0,224949.28125,233048.96875,233048.96875,1.0,1,0.0,0.0,10.0,"""Active""",0,0
1891521,126868,-1,453919.46875,450000,0.0,11547.0,0.0,11547.0,22924.890625,27000.0,27000.0,443044.40625,453919.46875,453919.46875,0.0,1,0.0,1.0,101.0,"""Active""",0,0


In [49]:
### FEATURE ENGINEERING

# logarithms
log_vars = ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_CURRENT",
            "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_INST_MIN_REGULARITY",
            "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "AMT_RECEIVABLE_PRINCIPAL",
            "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE"]
card = utils.create_logarithms(card, log_vars, replace = True)

In [51]:
# dummy encodnig for factors
card = card.to_dummies(cs.string(), drop_first = True)

In [52]:
### AGGREGATIONS
card_id = card.select(["SK_ID_CURR", "SK_ID_PREV"]).unique()

cnt_mon = (
    card
    .group_by("SK_ID_PREV")
    .agg(
        pl.col("MONTHS_BALANCE").count().alias("card_MON_COUNT")
    )
)

card = card.drop(["MONTHS_BALANCE","SK_ID_CURR"])

agg_card = utils.aggregate_data(card,id_var = "SK_ID_PREV")
agg_card = agg_card.join(cnt_mon,how="left",on="SK_ID_PREV")

agg_card = agg_card.join(card_id,how="left",on = "SK_ID_PREV")

agg_card = agg_card.drop("SK_ID_PREV")

agg_card = utils.aggregate_data(agg_card,id_var="SK_ID_CURR",label="card")


- Preparing the dataset...
- Extracted 0 factors and 25 numerics...
- Aggregating numeric features...
- Final dimensions: (104307, 101)
- Preparing the dataset...
- Extracted 0 factors and 101 numerics...
- Aggregating numeric features...
- Final dimensions: (103558, 405)


In [53]:
# count missings
nas = utils.count_missings(agg_card)
nas.head()

Feature,Total,Percent
str,i64,f64
"""card_AMT_DRAWINGS_ATM_CURRENT_…",103248,99.700651
"""card_AMT_DRAWINGS_OTHER_CURREN…",103248,99.700651
"""card_AMT_DRAWINGS_POS_CURRENT_…",103248,99.700651
"""card_CNT_DRAWINGS_ATM_CURRENT_…",103248,99.700651
"""card_CNT_DRAWINGS_OTHER_CURREN…",103248,99.700651


In [54]:
# check data
agg_card.head()

SK_ID_CURR,card_AMT_BALANCE_mean_mean,card_AMT_CREDIT_LIMIT_ACTUAL_mean_mean,card_AMT_DRAWINGS_ATM_CURRENT_mean_mean,card_AMT_DRAWINGS_CURRENT_mean_mean,card_AMT_DRAWINGS_OTHER_CURRENT_mean_mean,card_AMT_DRAWINGS_POS_CURRENT_mean_mean,card_AMT_INST_MIN_REGULARITY_mean_mean,card_AMT_PAYMENT_CURRENT_mean_mean,card_AMT_PAYMENT_TOTAL_CURRENT_mean_mean,card_AMT_RECEIVABLE_PRINCIPAL_mean_mean,card_AMT_RECIVABLE_mean_mean,card_AMT_TOTAL_RECEIVABLE_mean_mean,card_CNT_DRAWINGS_ATM_CURRENT_mean_mean,card_CNT_DRAWINGS_CURRENT_mean_mean,card_CNT_DRAWINGS_OTHER_CURRENT_mean_mean,card_CNT_DRAWINGS_POS_CURRENT_mean_mean,card_CNT_INSTALMENT_MATURE_CUM_mean_mean,card_NAME_CONTRACT_STATUS_Approved_mean_mean,card_NAME_CONTRACT_STATUS_Completed_mean_mean,card_NAME_CONTRACT_STATUS_Demand_mean_mean,card_NAME_CONTRACT_STATUS_Refused_mean_mean,card_NAME_CONTRACT_STATUS_Sent proposal_mean_mean,card_NAME_CONTRACT_STATUS_Signed_mean_mean,card_SK_DPD_mean_mean,card_SK_DPD_DEF_mean_mean,card_AMT_BALANCE_std_mean,card_AMT_CREDIT_LIMIT_ACTUAL_std_mean,card_AMT_DRAWINGS_ATM_CURRENT_std_mean,card_AMT_DRAWINGS_CURRENT_std_mean,card_AMT_DRAWINGS_OTHER_CURRENT_std_mean,card_AMT_DRAWINGS_POS_CURRENT_std_mean,card_AMT_INST_MIN_REGULARITY_std_mean,card_AMT_PAYMENT_CURRENT_std_mean,card_AMT_PAYMENT_TOTAL_CURRENT_std_mean,card_AMT_RECEIVABLE_PRINCIPAL_std_mean,card_AMT_RECIVABLE_std_mean,…,card_CNT_DRAWINGS_OTHER_CURRENT_min_max,card_CNT_DRAWINGS_POS_CURRENT_min_max,card_CNT_INSTALMENT_MATURE_CUM_min_max,card_NAME_CONTRACT_STATUS_Approved_min_max,card_NAME_CONTRACT_STATUS_Completed_min_max,card_NAME_CONTRACT_STATUS_Demand_min_max,card_NAME_CONTRACT_STATUS_Refused_min_max,card_NAME_CONTRACT_STATUS_Sent proposal_min_max,card_NAME_CONTRACT_STATUS_Signed_min_max,card_SK_DPD_min_max,card_SK_DPD_DEF_min_max,card_AMT_BALANCE_max_max,card_AMT_CREDIT_LIMIT_ACTUAL_max_max,card_AMT_DRAWINGS_ATM_CURRENT_max_max,card_AMT_DRAWINGS_CURRENT_max_max,card_AMT_DRAWINGS_OTHER_CURRENT_max_max,card_AMT_DRAWINGS_POS_CURRENT_max_max,card_AMT_INST_MIN_REGULARITY_max_max,card_AMT_PAYMENT_CURRENT_max_max,card_AMT_PAYMENT_TOTAL_CURRENT_max_max,card_AMT_RECEIVABLE_PRINCIPAL_max_max,card_AMT_RECIVABLE_max_max,card_AMT_TOTAL_RECEIVABLE_max_max,card_CNT_DRAWINGS_ATM_CURRENT_max_max,card_CNT_DRAWINGS_CURRENT_max_max,card_CNT_DRAWINGS_OTHER_CURRENT_max_max,card_CNT_DRAWINGS_POS_CURRENT_max_max,card_CNT_INSTALMENT_MATURE_CUM_max_max,card_NAME_CONTRACT_STATUS_Approved_max_max,card_NAME_CONTRACT_STATUS_Completed_max_max,card_NAME_CONTRACT_STATUS_Demand_max_max,card_NAME_CONTRACT_STATUS_Refused_max_max,card_NAME_CONTRACT_STATUS_Sent proposal_max_max,card_NAME_CONTRACT_STATUS_Signed_max_max,card_SK_DPD_max_max,card_SK_DPD_DEF_max_max,card_card_MON_COUNT_max
i32,f32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,u8,u8,u8,u8,u8,u8,i32,i32,f32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,f32,f32,f32,u8,u8,u8,u8,u8,u8,i32,i32,u32
222201,4.923712,12.100718,0.702847,0.688503,0.0,0.0,4.099162,8.779721,8.779721,4.85252,4.869211,4.869211,0.208333,0.204082,0.0,0.0,13.212766,0.0,0.0,0.0,0.0,0.0,0.0,0.326531,0.326531,5.67104,0.0,2.763232,2.73614,0.0,0.0,4.666999,1.355294,1.355294,5.687629,5.705861,…,0.0,0.0,1.0,0,0,0,0,0,0,0,0,12.162409,12.100718,12.125411,12.125411,0.0,0.0,9.441532,9.798183,9.798183,12.09528,12.127254,12.127254,6.0,6,0.0,0.0,21.0,0,0,0,0,0,0,8,8,49
168205,4.060626,11.598369,0.540926,0.540926,0.0,0.0,3.153138,8.009242,2.904479,3.951965,4.696315,4.696315,0.126437,0.126437,0.0,0.0,25.593023,0.0,0.0,0.0,0.0,0.0,0.0,0.022989,0.022989,5.503509,0.43813,2.229343,2.229343,0.0,0.0,4.226561,1.630553,4.249396,5.480888,5.453238,…,0.0,0.0,1.0,0,0,0,0,0,0,0,0,11.841813,11.813037,11.119899,11.119899,0.0,0.0,8.817447,11.166781,11.166781,11.794827,11.841813,11.841813,3.0,3,0.0,0.0,31.0,0,0,0,0,0,0,1,1,87
391680,9.418227,9.966509,2.146734,2.706872,0.0,0.934516,7.40613,7.054896,6.824245,9.276832,9.342846,9.

In [55]:
# clear memory
del card

In [56]:
# check card data
prev.head()

SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
i32,i32,str,f32,f32,f32,f32,f32,str,i32,str,i32,f32,f32,f32,str,str,i32,str,str,str,str,str,str,str,str,i32,str,f32,str,str,f32,f32,f32,f32,f32,f32
2030495,271877,"""Consumer loans""",1730.430054,17145.0,17145.0,0.0,17145.0,"""SATURDAY""",15,"""Y""",1,0.0,0.182832,0.867336,"""XAP""","""Approved""",-73,"""Cash through the bank""","""XAP""",null,"""Repeater""","""Mobile""","""POS""","""XNA""","""Country-wide""",35,"""Connectivity""",12.0,"""middle""","""POS mobile with interest""",365243.0,-42.0,300.0,-42.0,-37.0,0.0
2802425,108129,"""Cash loans""",25188.615234,607500.0,679671.0,null,607500.0,"""THURSDAY""",11,"""Y""",1,null,null,null,"""XNA""","""Approved""",-164,"""XNA""","""XAP""","""Unaccompanied""","""Repeater""","""XNA""","""Cash""","""x-sell""","""Contact center""",-1,"""XNA""",36.0,"""low_action""","""Cash X-Sell: low""",365243.0,-134.0,916.0,365243.0,365243.0,1.0
2523466,122040,"""Cash loans""",15060.735352,112500.0,136444.5,null,112500.0,"""TUESDAY""",11,"""Y""",1,null,null,null,"""XNA""","""Approved""",-301,"""Cash through the bank""","""XAP""","""Spouse, partner""","""Repeater""","""XNA""","""Cash""","""x-sell""","""Credit and cash offices""",-1,"""XNA""",12.0,"""high""","""Cash X-Sell: high""",365243.0,-271.0,59.0,365243.0,365243.0,1.0
2819243,176158,"""Cash loans""",47041.335938,450000.0,470790.0,null,450000.0,"""MONDAY""",7,"""Y""",1,null,null,null,"""XNA""","""Approved""",-512,"""Cash through the bank""","""XAP""",null,"""Repeater""","""XNA""","""Cash""","""x-sell""","""Credit and cash offices""",-1,"""XNA""",12.0,"""middle""","""Cash X-Sell: middle""",365243.0,-482.0,-152.0,-182.0,-177.0,1.0
1784265,202054,"""Cash loans""",31924.394531,337500.0,404055.0,null,337500.0,"""THURSDAY""",9,"""Y""",1,null,null,null,"""Repairs""","""Refused""",-781,"""Cash through the bank""","""HC""",null,"""Repeater""","""XNA""","""Cash""","""walk-in""","""Credit and cash offices""",-1,"""XNA""",24.0,"""high""","""Cash Street: high""",null,null,null,null,null,null


In [57]:
prev = (
    prev
    .with_columns(
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).alias("AMT_GIVEN_RATIO_1"),
        (pl.col("AMT_GOODS_PRICE") / pl.col("AMT_APPLICATION")).alias("AMT_GIVEN_RATIO_2"),
        (pl.col("AMT_DOWN_PAYMENT") / pl.col("AMT_APPLICATION")).alias("DOWN_PAYMENT_RATIO"),
        
        pl.len().over("SK_ID_CURR").alias("CNT_PREV_APPLICATIONS"),
        (pl.col("FLAG_LAST_APPL_PER_CONTRACT") == "Y").sum().over("SK_ID_CURR").alias("CNT_PREV_CONTRACTS"),
        
        pl.when(pl.col("WEEKDAY_APPR_PROCESS_START").is_in(["SATURDAY", "SUNDAY"]))
          .then(pl.lit("Weekend"))
          .otherwise(pl.lit("Working day"))
          .alias("DAY_APPR_PROCESS_START")
    )
    .with_columns(
        (pl.col("CNT_PREV_APPLICATIONS") / pl.col("CNT_PREV_CONTRACTS")).alias("APPL_PER_CONTRACT_RATIO")
    )
)

# Logarithms & Day Conversions
log_vars = ["AMT_CREDIT", "AMT_ANNUITY", "AMT_APPLICATION", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE"]
prev = utils.create_logarithms(prev, log_vars, replace=True)

day_vars = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", 
            "DAYS_LAST_DUE", "DAYS_TERMINATION", "DAYS_DECISION"]
prev = utils.convert_days(prev, day_vars, t=1, rounding=False, replace=True)

prev = prev.with_columns(
    (pl.col("DAYS_LAST_DUE_1ST_VERSION") - pl.col("DAYS_FIRST_DUE")).alias("DAYS_DUE_DIFF_1"),
    (pl.col("DAYS_LAST_DUE") - pl.col("DAYS_FIRST_DUE")).alias("DAYS_DUE_DIFF_2"),
    (pl.col("DAYS_TERMINATION") - pl.col("DAYS_FIRST_DRAWING")).alias("DAYS_TERMINATION_DIFF_1"),
    (pl.col("DAYS_TERMINATION") - pl.col("DAYS_FIRST_DUE")).alias("DAYS_TERMINATION_DIFF_2"),
    (pl.col("DAYS_TERMINATION") - pl.col("DAYS_LAST_DUE")).alias("DAYS_TERMINATION_DIFF_3"),
)

prev = utils.compute_accept_reject_ratio(prev, lags=[1, 3, 5])

drops = ["NAME_CLIENT_TYPE", "SK_ID_PREV"]
prev = prev.drop(drops)


In [58]:
# dummy encodnig for factors
prev = prev.to_dummies(cs.string(), drop_first = True)

In [59]:
# count missings
nas = utils.count_missings(prev)
nas.head()

Feature,Total,Percent
str,i64,f64
"""RATE_INTEREST_PRIMARY""",1664263,99.643698
"""RATE_INTEREST_PRIVILEGED""",1664263,99.643698
"""DAYS_TERMINATION_DIFF_1""",1661862,99.499944
"""DAYS_FIRST_DRAWING""",1607509,96.245691
"""DAYS_LAST_DUE_1ST_VERSION""",991321,59.352933


In [60]:

# 2. Aggregate data
agg_prev = utils.aggregate_data(prev, id_var="SK_ID_CURR", label="prev")

# 3. Clean up
omits = ["APPROVE_RATIO_1", "APPROVE_RATIO_3", "APPROVE_RATIO_5",  
         "REJECT_RATIO_1", "REJECT_RATIO_3",  "REJECT_RATIO_5", 
         "FLAG_LAST_APPL_PER_CONTRACT_Y", "CNT_PREV_CONTRACTS", "CNT_PREV_APPLICATIONS", 
         "APPL_PER_CONTRACT_RATIO"]

drop_cols = []
for var in omits:
    drop_cols.extend([f"prev_{var}_std", f"prev_{var}_min", f"prev_{var}_max"])


valid_drops = [col for col in drop_cols if col in agg_prev.columns]
agg_prev = agg_prev.drop(valid_drops)


- Preparing the dataset...
- Extracted 0 factors and 163 numerics...
- Aggregating numeric features...
- Final dimensions: (338857, 653)


In [61]:
# count missings
nas = utils.count_missings(agg_prev)
nas.head()

Feature,Total,Percent
str,i64,f64
"""prev_DAYS_TERMINATION_DIFF_1_s…",338827,99.991147
"""prev_DAYS_FIRST_DRAWING_std""",338656,99.940683
"""prev_RATE_INTEREST_PRIMARY_std""",338639,99.935666
"""prev_RATE_INTEREST_PRIVILEGED_…",338639,99.935666
"""prev_RATE_INTEREST_PRIMARY_mea…",333136,98.311677


In [62]:
# check data
agg_prev.head()

SK_ID_CURR,prev_NAME_CONTRACT_TYPE_Cash loans_mean,prev_NAME_CONTRACT_TYPE_Revolving loans_mean,prev_NAME_CONTRACT_TYPE_XNA_mean,prev_AMT_ANNUITY_mean,prev_AMT_APPLICATION_mean,prev_AMT_CREDIT_mean,prev_AMT_DOWN_PAYMENT_mean,prev_AMT_GOODS_PRICE_mean,prev_WEEKDAY_APPR_PROCESS_START_FRIDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_MONDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_SUNDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_THURSDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_TUESDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_WEDNESDAY_mean,prev_HOUR_APPR_PROCESS_START_mean,prev_FLAG_LAST_APPL_PER_CONTRACT_N_mean,prev_NFLAG_LAST_APPL_IN_DAY_mean,prev_RATE_DOWN_PAYMENT_mean,prev_RATE_INTEREST_PRIMARY_mean,prev_RATE_INTEREST_PRIVILEGED_mean,prev_NAME_CASH_LOAN_PURPOSE_Building a house or an annex_mean,prev_NAME_CASH_LOAN_PURPOSE_Business development_mean,prev_NAME_CASH_LOAN_PURPOSE_Buying a garage_mean,prev_NAME_CASH_LOAN_PURPOSE_Buying a holiday home / land_mean,prev_NAME_CASH_LOAN_PURPOSE_Buying a home_mean,prev_NAME_CASH_LOAN_PURPOSE_Buying a new car_mean,prev_NAME_CASH_LOAN_PURPOSE_Buying a used car_mean,prev_NAME_CASH_LOAN_PURPOSE_Car repairs_mean,prev_NAME_CASH_LOAN_PURPOSE_Education_mean,prev_NAME_CASH_LOAN_PURPOSE_Everyday expenses_mean,prev_NAME_CASH_LOAN_PURPOSE_Furniture_mean,prev_NAME_CASH_LOAN_PURPOSE_Gasification / water supply_mean,prev_NAME_CASH_LOAN_PURPOSE_Hobby_mean,prev_NAME_CASH_LOAN_PURPOSE_Journey_mean,prev_NAME_CASH_LOAN_PURPOSE_Medicine_mean,prev_NAME_CASH_LOAN_PURPOSE_Money for a third person_mean,…,prev_CNT_PAYMENT_max,prev_NAME_YIELD_GROUP_XNA_max,prev_NAME_YIELD_GROUP_high_max,prev_NAME_YIELD_GROUP_low_action_max,prev_NAME_YIELD_GROUP_low_normal_max,prev_PRODUCT_COMBINATION_Card Street_max,prev_PRODUCT_COMBINATION_Card X-Sell_max,prev_PRODUCT_COMBINATION_Cash_max,prev_PRODUCT_COMBINATION_Cash Street: high_max,prev_PRODUCT_COMBINATION_Cash Street: low_max,prev_PRODUCT_COMBINATION_Cash Street: middle_max,prev_PRODUCT_COMBINATION_Cash X-Sell: high_max,prev_PRODUCT_COMBINATION_Cash X-Sell: low_max,prev_PRODUCT_COMBINATION_Cash X-Sell: middle_max,prev_PRODUCT_COMBINATION_POS household with interest_max,prev_PRODUCT_COMBINATION_POS household without interest_max,prev_PRODUCT_COMBINATION_POS industry with interest_max,prev_PRODUCT_COMBINATION_POS industry without interest_max,prev_PRODUCT_COMBINATION_POS mobile without interest_max,prev_PRODUCT_COMBINATION_POS other with interest_max,prev_PRODUCT_COMBINATION_POS others without interest_max,prev_PRODUCT_COMBINATION_null_max,prev_DAYS_FIRST_DRAWING_max,prev_DAYS_FIRST_DUE_max,prev_DAYS_LAST_DUE_1ST_VERSION_max,prev_DAYS_LAST_DUE_max,prev_DAYS_TERMINATION_max,prev_NFLAG_INSURED_ON_APPROVAL_max,prev_AMT_GIVEN_RATIO_1_max,prev_AMT_GIVEN_RATIO_2_max,prev_DOWN_PAYMENT_RATIO_max,prev_DAY_APPR_PROCESS_START_Working day_max,prev_DAYS_DUE_DIFF_1_max,prev_DAYS_DUE_DIFF_2_max,prev_DAYS_TERMINATION_DIFF_1_max,prev_DAYS_TERMINATION_DIFF_2_max,prev_DAYS_TERMINATION_DIFF_3_max
i32,f64,f64,f64,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,f32,f32,f32,f32,f32,f32,f32,f32,u8,f32,f32,f32,f32,f32
354068,0.0,0.0,0.0,8.037987,10.935899,11.195932,0.0,10.935899,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,1.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,36.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,null,2104.0,1054.0,1054.0,1049.0,0.0,1.296979,1.0,0.0,0,-1050.0,-1050.0,null,-1055.0,-5.0
455796,0.75,0.0,0.0,8.611407,2.588263,2.574772,8.051183,10.35305,0.25,0.25,0.0,0.0,0.0,0.0,12.0,0.0,1.0,0.104014,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,6.0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,null,475.0,325.0,355.0,353.0,0.0,0.947467,1.0,0.100043,1,-150.0,-120.0,null,-122.0,-2.0
425883,0.0,0.0,0.0,9.50042,11.66913,11.769505,0.0,11.66913,0.0,0.0,1.0,0.0,0.0,0.0,10.0,0.0,1.0,0.0,null,null,0.0,0.0,0.0,0.0,0

In [63]:
# clear memory
del prev

In [64]:
# Merge data sequentially and delete the individual tables to save RAM
print("Initial shape:", appl.shape)

appl = appl.join(agg_buro, on="SK_ID_CURR", how="left")
del agg_buro
print("After buro:", appl.shape)

appl = appl.join(agg_prev, on="SK_ID_CURR", how="left")
del agg_prev
print("After prev:", appl.shape)

appl = appl.join(agg_inst, on="SK_ID_CURR", how="left")
del agg_inst
print("After inst:", appl.shape)

appl = appl.join(agg_poca, on="SK_ID_CURR", how="left")
del agg_poca
print("After poca:", appl.shape)

appl = appl.join(agg_card, on="SK_ID_CURR", how="left")
del agg_card
print("Final shape:", appl.shape)


Initial shape: (356255, 109)
After buro: (356255, 335)
After prev: (356255, 960)
After inst: (356255, 1108)
After poca: (356255, 1320)
Final shape: (356255, 1724)


In [67]:
check_cols = [
    "AMT_ANNUITY", "prev_AMT_ANNUITY_mean", 
    "AMT_CREDIT", "prev_AMT_CREDIT_mean", 
    "AMT_GOODS_PRICE", "prev_AMT_GOODS_PRICE_mean", 
    "buro_AMT_ANNUITY_mean", "buro_AMT_CREDIT_SUM_mean"
]

for col in check_cols:
    print(f"Has '{col}'? : {col in appl.columns}")


Has 'AMT_ANNUITY'? : True
Has 'prev_AMT_ANNUITY_mean'? : True
Has 'AMT_CREDIT'? : True
Has 'prev_AMT_CREDIT_mean'? : True
Has 'AMT_GOODS_PRICE'? : True
Has 'prev_AMT_GOODS_PRICE_mean'? : True
Has 'buro_AMT_ANNUITY_mean'? : True
Has 'buro_AMT_CREDIT_SUM_mean'? : True


In [68]:
appl = (
    appl
    .with_columns(
        (pl.col("AMT_ANNUITY") / pl.col("prev_AMT_ANNUITY_mean")).alias("mix_AMT_PREV_ANNUITY_RATIO"),
        (pl.col("AMT_CREDIT") / pl.col("prev_AMT_CREDIT_mean")).alias("mix_AMT_PREV_CREDIT_RATIO"),
        (pl.col("AMT_GOODS_PRICE") / pl.col("prev_AMT_GOODS_PRICE_mean")).alias("mix_AMT_PREV_GOODS_PRICE_RATIO"),
        (pl.col("AMT_ANNUITY") / pl.col("buro_AMT_ANNUITY_mean")).alias("mix_AMT_BURO_ANNUITY_RATIO"),
        (pl.col("AMT_CREDIT") / pl.col("buro_AMT_CREDIT_SUM_mean")).alias("mix_AMT_BURO_CREDIT_RATIO")
    )
)


In [69]:
# dummy encodnig for factors
appl = appl.to_dummies(cs.string(), drop_first = True)

In [70]:
# count missings
nas = utils.count_missings(appl)
nas.head()

Feature,Total,Percent
str,i64,f64
"""prev_DAYS_TERMINATION_DIFF_1_s…",356225,99.991579
"""prev_DAYS_FIRST_DRAWING_std""",356054,99.94358
"""prev_RATE_INTEREST_PRIMARY_std""",356037,99.938808
"""prev_RATE_INTEREST_PRIVILEGED_…",356037,99.938808
"""card_AMT_DRAWINGS_ATM_CURRENT_…",355945,99.912984


In [71]:
# partitioning
train = appl.filter(pl.col("SK_ID_CURR").is_in(y["SK_ID_CURR"]))
test = appl.filter(~pl.col("SK_ID_CURR").is_in(y["SK_ID_CURR"]))

# clear memory
del appl


In [72]:
# check dimensions
print(train.shape)
print(test.shape)

(307511, 1843)
(48744, 1843)


In [73]:
# export CSV
train.write_parquet("../data/processed/train_full_cor.parquet",)
test.write_parquet("../data/processed/test_full_cor.parquet",)
y.write_parquet("../data/processed/y_full_cor.parquet",)